In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

from datetime import datetime
from zoneinfo import ZoneInfo

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [2]:
# Simple mock dataset (extend or swap with your API later)
MOCK_WEATHER: dict[str, dict[str, str]] = {
    "mumbai": {
        "condition": "Humid, partly cloudy",
        "temp_c": "31",
        "humidity": "75%",
        "rain_chance": "40%",
    },
    "delhi": {
        "condition": "Hot, dry",
        "temp_c": "34",
        "humidity": "35%",
        "rain_chance": "10%",
    },
    "hyderabad": {
        "condition": "Warm, breezy",
        "temp_c": "30",
        "humidity": "55%",
        "rain_chance": "20%",
    },
    "bengaluru": {
        "condition": "Mild, cloudy",
        "temp_c": "26",
        "humidity": "65%",
        "rain_chance": "30%",
    },
}

In [3]:
def _now_ist_str() -> str:
    ist = ZoneInfo("Asia/Kolkata")
    return datetime.now(ist).strftime("%Y-%m-%d %H:%M IST")

In [4]:
@tool
def get_weather(city: str) -> str:
    """Return a short, human-friendly weather report for the given city (mock data)."""
    if not city or not city.strip():
        return "Please provide a city name."
    key = city.strip().lower()
    data = MOCK_WEATHER.get(key)
    if not data:
        # Sensible default when city isn't in our mock set
        data = {
            "condition": "Partly cloudy",
            "temp_c": "30",
            "humidity": "60%",
            "rain_chance": "20%",
        }
    return (
        f"City: {city.strip().title()}\n"
        f"Time: {_now_ist_str()}\n"
        f"Condition: {data['condition']}\n"
        f"Temperature: {data['temp_c']} °C\n"
        f"Humidity: {data['humidity']}\n"
        f"Chance of rain: {data['rain_chance']}"
    )

In [ ]:
def build_weather_agent():
    llm = ChatOpenAI(
        model=os.getenv("GPT_MODEL", "your-gpt-model"),
        base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
        api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
    )
    tools = [get_weather]

    SYSTEM_PROMPT = (
        "You are a helpful weather assistant.\n"
        "RULES:\n"
        "1) Always call the get_weather tool to answer weather questions.\n"
        "2) If the city is missing or unclear, ask the user to specify a city.\n"
        "3) Return concise, friendly answers.\n"
    )

    return create_agent(llm, tools=tools, system_prompt=SYSTEM_PROMPT)


weather_agent = build_weather_agent()


def ask(agent, query: str) -> str:
    return agent.invoke({"messages": [("human", query)]})["messages"][-1].content


print(" Weather agent ready.")

 Weather agent ready.


In [10]:
# Known city
print(ask(weather_agent, "What’s the weather in Mumbai?"))

# Another known city
print(ask(weather_agent, "Weather update for Bengaluru, please."))

# Unknown or missing city → agent should ask to clarify
print(ask(weather_agent, "What’s the weather like today?"))

Currently in Mumbai (IST 01:43): Humid, partly cloudy. Temperature 31°C, humidity 75%. Rain chance 40%. If you’re heading out, you might want to carry a light umbrella.
Here’s the latest for Bengaluru (as of 01:43 IST):
- Condition: Mild, cloudy
- Temperature: 26°C
- Humidity: 65%
- Chance of rain: 30%

A light jacket or umbrella might be handy if you're heading out. Would you like a forecast for the rest of the day?
Sure—Which city would you like the weather for today?
